# SpectraMan — Fourier Transform Spectrum Processor

Upload a spectrum file (`.txt`, `.csv`, `.dat`, `.asc`, `.jdx`, `.dx`) below and use the controls to explore its FFT, apply apodization/deconvolution windows, and export the processed signal.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq, ifft
from scipy.signal import windows
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io, os
import base64


In [ ]:
# Configuration settings for spectrum processing
class ProcessConfig:
    def __init__(self, normalize_range=None, apply_log = False):
        self.normalize_range = normalize_range
        self.apply_log = apply_log

In [ ]:
# Container for FFT output
class FFTData:
    def __init__(self, canal, values):
        self.canal = canal
        self.values = values

In [ ]:
class SpectrumData:
    def __init__(self, x, y, config: ProcessConfig):
        self.x = x
        self.y = y
        self.config = config

        self.fft = None
        self.current_fft = None
        self.log_fft = None
        self.ifft = None

        self.window_used = None
        self.deconv_filter = None

    def preprocess(self):
        self.x_raw = self.x.copy()
        self.y_raw = self.y.copy()

        x_uniform = np.linspace(self.x.min(), self.x.max(), len(self.x))
        y_uniform = np.interp(x_uniform, self.x, self.y)

        y_uniform -= np.mean(y_uniform)

        self.x = x_uniform
        self.y = y_uniform

    def compute_fft(self):
        n = len(self.y)
        dx = np.mean(np.diff(self.x))
        canal = fftfreq(n, d=dx)          # fix 1: was fftcanal
        values = fft(self.y)

        self.fft = FFTData(canal, values)
        self.current_fft = FFTData(canal.copy(), values.copy())

    def compute_log_fft(self):
        if self.fft is None:
            raise ValueError("Run compute_fft first")

        amp = np.abs(self.fft.values)
        amp_ref = np.max(amp)

        if amp_ref == 0:
            raise ValueError("FFT is all zeros")

        ratio = np.maximum(amp / amp_ref, 1e-12)
        self.log_fft = (10 * np.log10(ratio))

    def apodize(self, window_type='Gaussian', L=None, sigma=None):
        if self.current_fft is None:
            raise ValueError("Run compute_fft first")

        canal = self.current_fft.canal
        window = self.build_apod_window(canal, window_type, L, sigma)

        self.window_used = window
        self.current_fft = FFTData(canal, self.current_fft.values * window)

    def deconvolve_with_apodization(
        self,
        delta=0.01,
        rho=1.5,
        window_type='Gaussian',
        sigma=None,
        L=None,
    ):
        if self.fft is None:
            raise ValueError("Run compute_fft first")

        canal = self.fft.canal
        canal_abs = np.abs(canal)

        H = np.exp(-delta * (canal_abs ** rho))
        eps = 1e-12
        deconv_filter = 1.0 / (H + eps)

        apod_window = self.build_apod_window(canal, window_type, L, sigma)

        combined_filter = deconv_filter * apod_window

        self.deconv_filter = deconv_filter
        self.window_used = apod_window
        self.current_fft = FFTData(canal, self.fft.values * combined_filter)  # fix 2: was freq

    def build_apod_window(self, canal, window_type, L, sigma):
        canal_abs = np.abs(canal)
        canal_max = np.max(canal_abs)

        if L is None:
            if sigma is None:
                sigma = 0.10
            L = sigma * canal_max

        if window_type == 'Boxcar':
            window = self.apod_boxcar(canal_abs, L)
        elif window_type == 'Gaussian':
            window = self.apod_gaussian(canal_abs, L)
        elif window_type == 'Sigmoid':
            window = self.apod_sigmoid(canal_abs, L)
        elif window_type == 'Bessel':
            window = self.apod_bessel(canal_abs, L)
        elif window_type == 'Sync2':
            window = self.apod_sync2(canal_abs, L)
        else:
            raise ValueError(
                f"Unknown window type '{window_type}'. "
                "Choose from: Boxcar, Gaussian, Sigmoid, Bessel, Sync2"
            )

        window[0] = 1.0
        return window

    @staticmethod
    def apod_boxcar(canal_abs, L):
        """D(u, L) = 1 if u < L, else 0"""
        return np.where(canal_abs < L, 1.0, 0.0)

    @staticmethod
    def apod_gaussian(canal_abs, L):
        """D(u, L) = exp(-u² / L²)"""
        return np.exp(-(canal_abs ** 2) / (L ** 2))

    @staticmethod
    def apod_sigmoid(canal_abs, L, b=10.0):
        """D(u, L) = 1 / (1 + exp(b * (u - L)))"""
        return 1.0 / (1.0 + np.exp(b * (canal_abs - L)))

    @staticmethod
    def apod_bessel(canal_abs, L):           # fix 3: parameter was freq_abs, used canal_abs
        """D(u, L) = (1 - u²/L²)²  if u < L, else 0"""
        return np.where(canal_abs < L, (1.0 - (canal_abs / L) ** 2) ** 2, 0.0)

    @staticmethod
    def apod_sync2(canal_abs, L):            # fix 4: parameter was inconsistently named
        """D(u, L) = sinc²(a*u) / (a*u)²  if u < L, else 0  where a = π/L"""
        a = np.pi / L
        au = a * canal_abs
        with np.errstate(invalid='ignore', divide='ignore'):
            window = np.where(au == 0, 1.0, (np.sin(au) / au) ** 2)
        window = np.where(canal_abs < L, window, 0.0)
        return window

    def compute_ifft(self):
        if self.current_fft is None:
            raise ValueError("No FFT available")

        self.ifft = np.real(ifft(self.current_fft.values))

In [ ]:
class PlotDiagrams:

    @staticmethod
    def plot_spectrum(x, y, title, save=False):
        plt.figure(figsize=(12, 4))
        plt.plot(x, y, lw=0.8)

        plt.title(title)
        plt.xlabel("Wavenumber (cm⁻¹)")
        plt.ylabel("Intensity (a.u.)")

        plt.grid()

        if save:
            plt.savefig(f"{title}.png", dpi=150, bbox_inches="tight")

        plt.show()

    @staticmethod
    def plot_fft(fft_data, title="FFT Spectrum", save=False):
        mask = fft_data.canal > 0

        plt.figure(figsize=(12, 4))
        plt.plot(fft_data.canal[mask], np.abs(fft_data.values[mask]), lw=0.8)

        plt.title(title)
        plt.xlabel("OPD / Canal (cm)")
        plt.ylabel("Amplitude")
        plt.grid()

        if save:
            plt.savefig(f"{title}.png", dpi=150, bbox_inches="tight")

        plt.show()

    @staticmethod
    def plot_log_fft(spec, show_apodization=False, show_combined=False, save=False):
        if spec.log_fft is None:
            raise ValueError("Run compute_log_fft first")

        mask = spec.fft.canal > 0
        canal = spec.fft.canal[mask]

        plt.figure(figsize=(12, 4))
        plt.plot(canal, spec.log_fft[mask], lw=0.8, label="Log FFT")

        if show_apodization:
            if spec.window_used is None:
                raise ValueError("Run apodize / apply_window first")

            window_db = 10 * np.log10(np.maximum(spec.window_used[mask], 1e-12))
            plt.plot(
                canal,
                window_db,
                lw=1.2,
                linestyle="--",
                label="Apodization Window"
            )

        if show_combined:
            if spec.deconv_filter is None or spec.window_used is None:
                raise ValueError(
                    "Run deconvolve_with_apodization first — "
                    "both deconv_filter and window_used must be set"
                )

            EPS = 1e-6

            deconv_db = 20 * np.log10(np.maximum(spec.deconv_filter[mask], EPS))
            plt.plot(
                canal,
                deconv_db,
                lw=1.2,
                linestyle=":",
                label="Deconvolution Filter"
            )

            window_db = 20 * np.log10(np.maximum(spec.window_used[mask], EPS))
            plt.plot(
                canal,
                window_db,
                lw=1.2,
                linestyle="--",
                label="Apodization Window"
            )

            combined = spec.deconv_filter[mask] * spec.window_used[mask]
            combined_db = 20 * np.log10(np.maximum(combined, EPS))
            plt.plot(
                canal,
                combined_db,
                lw=1.5,
                linestyle="-.",
                label="Combined Filter (Deconv × Apod)"
            )

            # Fit y-axis to the signal range so crossings are the focus
            signal_db = spec.log_fft[mask]
            sig_min = np.min(signal_db)
            sig_max = np.max(signal_db)
            padding = (sig_max - sig_min) * 0.5
            plt.ylim(sig_min - padding, sig_max + padding)

        plt.title("Log FFT")
        plt.xlabel("OPD / Canal (cm)")
        plt.ylabel("Amplitude (dB)")
        plt.legend()
        plt.grid()

        if save:
            plt.savefig("log_fft.png", dpi=150, bbox_inches="tight")

        plt.show()

    @staticmethod
    def plot_processed_fft(spec, title="Processed FFT", save=False):
        if spec.current_fft is None:
            raise ValueError("No processed FFT available")

        mask = spec.current_fft.canal > 0

        plt.figure(figsize=(12, 4))
        plt.plot(
            spec.current_fft.canal[mask],
            np.abs(spec.current_fft.values[mask]),
            lw=0.8
        )

        plt.title(title)
        plt.xlabel("OPD / Canal (cm)")
        plt.ylabel("Amplitude")
        plt.grid()

        if save:
            plt.savefig(f"{title}.png", dpi=150, bbox_inches="tight")

        plt.show()

    @staticmethod
    def plot_fft_before_after_processing(spec, title="FFT Before and After Processing", save=False):
        if spec.fft is None:
            raise ValueError("Run compute_fft first")

        if spec.current_fft is None:
            raise ValueError("No processed FFT available")

        mask = spec.fft.canal > 0
        canal = spec.fft.canal[mask]
        fft_original = np.abs(spec.fft.values[mask])
        fft_processed = np.abs(spec.current_fft.values[mask])

        plt.figure(figsize=(12, 4))
        plt.plot(canal, fft_original, label="Original FFT", lw=0.8)
        plt.plot(canal, fft_processed, label="Processed FFT", lw=0.8)

        plt.title(title)
        plt.xlabel("OPD / Canal (cm)")
        plt.ylabel("Amplitude")
        plt.legend()
        plt.grid()

        if save:
            plt.savefig(f"{title}.png", dpi=150, bbox_inches="tight")

        plt.show()

    @staticmethod
    def plot_processing_difference(spec, title="Difference: Processed FFT - Original FFT", save=False):
        if spec.fft is None:
            raise ValueError("Run compute_fft first")

        if spec.current_fft is None:
            raise ValueError("No processed FFT available")

        mask = spec.fft.canal > 0
        canal = spec.fft.canal[mask]

        diff = np.abs(spec.current_fft.values[mask]) - np.abs(spec.fft.values[mask])

        plt.figure(figsize=(12, 4))
        plt.plot(canal, diff, lw=0.8)
        plt.axhline(0, color="black", linestyle="--", linewidth=0.8)

        plt.title(title)
        plt.xlabel("OPD / Canal (cm)")
        plt.ylabel("Amplitude Difference")
        plt.grid()

        if save:
            plt.savefig(f"{title}.png", dpi=150, bbox_inches="tight")

        plt.show()

    @staticmethod
    def plot_ifft(spec, save=False):
        if spec.ifft is None:
            raise ValueError("Run compute_ifft first")

        plt.figure(figsize=(12, 4))
        plt.plot(spec.x, spec.ifft, lw=0.8)

        plt.title("Inverse FFT")
        plt.xlabel("Wavenumber (cm⁻¹)")
        plt.ylabel("Intensity (a.u.)")
        plt.grid()

        if save:
            plt.savefig("Inverse_FFT.png", dpi=150, bbox_inches="tight")

        plt.show()

    @staticmethod
    def plot_original_vs_ifft(spec, title="Original vs Processed Spectrum"):
        if spec.ifft is None:
            raise ValueError("Run compute_ifft first")

        plt.figure(figsize=(12, 4))
        plt.plot(spec.x, spec.y, label="Original", lw=0.8)
        plt.plot(spec.x, spec.ifft, label="Processed", lw=0.8)

        plt.title(title)
        plt.xlabel("Wavenumber (cm⁻¹)")
        plt.ylabel("Intensity (a.u.)")
        plt.legend()
        plt.grid()

        plt.show()

    @staticmethod
    def plot_ifft_difference(spec, title="Difference: Original - Processed Spectrum"):
        if spec.ifft is None:
            raise ValueError("Run compute_ifft first")

        diff = spec.y - spec.ifft

        plt.figure(figsize=(12, 4))
        plt.plot(spec.x, diff, lw=0.8)
        plt.axhline(0, color="black", linestyle="--", linewidth=0.8)

        plt.title(title)
        plt.xlabel("Wavenumber (cm⁻¹)")
        plt.ylabel("Residual Intensity")
        plt.grid()

        plt.show()

In [ ]:
# Spectrum File Loader .txt / .csv / .dat / .asc / .jdx / .dx

import re

def parse_jdx(raw_bytes: bytes):
    """Parse a JCAMP-DX file and return (x, y) numpy arrays."""
    text = raw_bytes.decode('latin-1')
    lines = text.splitlines()

    header = {}
    for ln in lines:
        m = re.match(r'##([^=]+)=(.+)', ln)
        if m:
            header[m.group(1).strip().upper()] = m.group(2).strip()

    xfactor = float(header.get('XFACTOR', 1.0))
    yfactor = float(header.get('YFACTOR', 1.0))
    firstx  = float(header.get('FIRSTX',  0.0))
    lastx   = float(header.get('LASTX',   1.0))
    deltax  = float(header.get('DELTAX',  1.0)) if 'DELTAX' in header else None
    npoints = int(header.get('NPOINTS',   0))
    data_form = header.get('XYDATA', header.get('XYPOINTS', '')).upper()

    in_data = False
    data_lines = []
    for ln in lines:
        tag = ln.strip().upper()
        if tag.startswith('##XYDATA') or tag.startswith('##XYPOINTS'):
            in_data = True
            continue
        if in_data:
            if tag.startswith('##END') or tag.startswith('##'):
                break
            data_lines.append(ln.strip())

    if '(X++(Y..Y))' in data_form or not data_form:
        xs, ys = [], []
        for ln in data_lines:
            if not ln:
                continue
            tokens = ln.split()
            if not tokens:
                continue
            x_start = float(tokens[0]) * xfactor
            y_vals  = [float(v) * yfactor for v in tokens[1:]]
            if deltax is not None and len(y_vals) > 1:
                x_row = [x_start + i * deltax * xfactor for i in range(len(y_vals))]
            else:
                x_row = [x_start] * len(y_vals)
            xs.extend(x_row)
            ys.extend(y_vals)
        x_arr = np.array(xs[:npoints] if npoints else xs)
        y_arr = np.array(ys[:npoints] if npoints else ys)

    elif '(XY..XY)' in data_form:
        xs, ys = [], []
        for ln in data_lines:
            parts = re.split(r'[,\s]+', ln.strip())
            parts = [p for p in parts if p]
            for i in range(0, len(parts) - 1, 2):
                xs.append(float(parts[i])   * xfactor)
                ys.append(float(parts[i+1]) * yfactor)
        x_arr = np.array(xs)
        y_arr = np.array(ys)
    else:
        raise ValueError(f"Unsupported JCAMP-DX data form: '{data_form}'")

    if len(x_arr) > 1 and x_arr[0] > x_arr[-1]:
        x_arr = x_arr[::-1]
        y_arr = y_arr[::-1]

    metadata = {
        'title':    header.get('TITLE', ''),
        'xunits':   header.get('XUNITS', ''),
        'yunits':   header.get('YUNITS', ''),
        'data_type': header.get('DATA TYPE', ''),
    }
    return x_arr, y_arr, metadata

def load_spectrum(raw_bytes: bytes, filename: str):
    ext = filename.rsplit('.', 1)[-1].lower() if '.' in filename else ''

    if ext in ('jdx', 'dx'):
        return parse_jdx(raw_bytes)

    try:
        data = np.loadtxt(io.BytesIO(raw_bytes))
    except Exception:
        # Try comma-separated
        data = np.loadtxt(io.BytesIO(raw_bytes), delimiter=',')
    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError("Expected a two-column file (wavenumber, intensity).")
    x_arr, y_arr = data[:, 0], data[:, 1]
    metadata = {'title': filename, 'xunits': '', 'yunits': '', 'data_type': 'TEXT'}
    return x_arr, y_arr, metadata

def detect_transmittance_by_shape(y, percent_threshold=1.5):
    """
    Guess whether y-data is transmittance or absorbance, using shape alone
    (no YUNITS keyword needed).

    Logic:
      - Transmittance baseline sits near the top of its range, with peaks
        dipping down -> most points lie in the upper half of [min, max].
      - Absorbance baseline sits near the bottom, with peaks rising up
        -> most points lie in the lower half of [min, max].
      - As a sanity check, absorbance is unbounded above; a max value far
        beyond typical transmittance bounds (~1 or ~100) is treated as
        strong evidence it's already absorbance, regardless of shape.

    Returns
    -------
    is_transmittance : bool
    confidence : float   # fraction of points supporting the verdict (0.5-1.0)
    """
    y = np.asarray(y, dtype=float)
    y_min, y_max = np.nanmin(y), np.nanmax(y)

    if y_max - y_min < 1e-12:
        return False, 0.5  # flat/degenerate data, can't tell

    midpoint = (y_min + y_max) / 2.0
    frac_above_mid = np.mean(y > midpoint)

    # Strong override: absorbance has no fixed upper bound, so a max far
    # beyond 1 (fractional) or 100 (percent) is essentially never transmittance.
    if y_max > 3.5 * percent_threshold and y_max > 3.5:
        return False, max(frac_above_mid, 1 - frac_above_mid)

    is_transmittance = frac_above_mid > 0.5
    confidence = frac_above_mid if is_transmittance else (1 - frac_above_mid)

    return is_transmittance, confidence

def transmittance_to_absorbance(y):
    """
    Convert transmittance data to absorbance: A = -log10(T).
    Auto-detects whether y is on a fractional (0-1) or percent (0-100) scale.
    """
    y = np.asarray(y, dtype=float)
    y_max = np.nanmax(y)

    T = y / 100.0 if y_max > 3.5 else y.copy()
    T = np.clip(T, 1e-6, None)  # guard against log(0)/negative values

    return -np.log10(T)

def ensure_absorbance(y, label=""):
    """
    Check y-data with detect_transmittance_by_shape and convert it to
    absorbance if it looks like transmittance. Returns the (possibly
    converted) y array.
    """
    is_transmittance, confidence = detect_transmittance_by_shape(y)
    if is_transmittance:
        print(f"[{label}] Detected transmittance data (confidence={confidence:.2f}) "
              "-> converting to absorbance.")
        y = transmittance_to_absorbance(y)
    return y

In [ ]:
class SpectrumGUI:
    WINDOWS = ["Gaussian", "Boxcar", "Sigmoid", "Bessel", "Sync2"]

    def __init__(self):
        self.spec = None
        self.mode = "apod"          # "apod" or "deconv"
        self._applied = False       # True once Apply has committed a processing
        self._busy = False          # guards bulk toggle resets
        self._panel_open = True     # Normalized Domain panel visibility and(Close button)

        # File loader
        self.out_loader = widgets.Output()
        self.btn_clear = widgets.Button(description="Clear File", button_style="danger",
                                        icon="trash", disabled=True)
        self.lbl_file = widgets.Label("No file loaded")

        # Base spectrum views
        self.tg_orig = widgets.ToggleButton(description="Original Spectra")
        self.tg_fft = widgets.ToggleButton(description="FFT")
        self.tg_lfft = widgets.ToggleButton(description="LOG FFT")

        # Normalized Domain toolbar (Apply | Br. Point | Window | Close)
        self.btn_apply = widgets.Button(description="Apply", button_style="success",
                                        layout=widgets.Layout(width="90px"))
        self.tg_brpoint = widgets.ToggleButton(description="Brеак Point",
                                        layout=widgets.Layout(width="90px"))
        self.dd_win = widgets.Dropdown(options=self.WINDOWS, value="Gaussian",
                                       layout=widgets.Layout(width="150px"))
        self.btn_close = widgets.Button(description="Close",
                                        layout=widgets.Layout(width="80px"))

        # Mode selector
        self.rb_mode = widgets.RadioButtons(
            options=[("Apodization", "apod"),
                     ("Deconvolution + Apodization", "deconv")],
            value="apod", description="Mode:",
            style={"description_width": "60px"})

        # Sliders: sigma (bottom, horizontal); delta & rho (right, vertical).
        self.sl_sigma = widgets.FloatSlider(value=0.64, min=0.01, max=1.0, step=0.01,
                                            description="sigma:", readout_format=".2f",
                                            continuous_update=False,
                                            layout=widgets.Layout(width="620px"))
        self.sl_delta = widgets.FloatSlider(value=65.0, min=0.0, max=200.0, step=1.0,
                                            description="delta", readout_format=".0f",
                                            orientation="vertical", continuous_update=False,
                                            layout=widgets.Layout(height="150px"))
        self.sl_rho = widgets.FloatSlider(value=1.65, min=0.5, max=3.0, step=0.05,
                                          description="rho", readout_format=".2f",
                                          orientation="vertical", continuous_update=False,
                                          layout=widgets.Layout(height="150px"))
        self.lbl_apod = widgets.Label("Apod.Point = 64%")

        # Modified FFT views
        W = lambda px: widgets.Layout(width=px)
        self.tg_mfft = widgets.ToggleButton(description="Show Modified FFT Signal",
                                            disabled=True, layout=W("260px"))
        self.tg_cmpfft = widgets.ToggleButton(description="Show Original vs Modified FFT Signal",
                                              disabled=True, layout=W("320px"))
        self.tg_difffft = widgets.ToggleButton(description="Show Difference (FFT)",
                                               disabled=True, layout=W("220px"))
        # Processed signal views
        self.tg_msig = widgets.ToggleButton(description="Show Modified Signal",
                                            disabled=True, layout=W("260px"))
        self.tg_cmpsig = widgets.ToggleButton(description="Show Original and Modified Signal",
                                             disabled=True, layout=W("320px"))
        self.tg_diffsig = widgets.ToggleButton(description="Show Difference (Signal)",
                                               disabled=True, layout=W("220px"))

        self.btn_export = widgets.Button(description="Export Signal", button_style="warning",
                                         icon="download", disabled=True)
        self.out_norm = widgets.Output(layout=widgets.Layout(width="720px", min_height="320px"))
        self.out_status = widgets.Output()
        self.out_plot = widgets.Output()
        self.out_download = widgets.Output()

        # Toggle groups
        self._base_toggles = [self.tg_orig, self.tg_fft, self.tg_lfft]
        self._proc_toggles = [self.tg_mfft, self.tg_cmpfft, self.tg_difffft,
                              self.tg_msig, self.tg_cmpsig, self.tg_diffsig]
        self._all_toggles = self._base_toggles + self._proc_toggles

        # Events
        self.btn_clear.on_click(self._clear)
        self.btn_apply.on_click(self._apply)
        self.btn_close.on_click(self._toggle_panel)
        self.btn_export.on_click(self._export)
        self.rb_mode.observe(lambda c: self._on_mode_change(c["new"]), names="value")
        self.dd_win.observe(lambda c: self._on_window_change(), names="value")
        for s in (self.sl_sigma, self.sl_delta, self.sl_rho):
            s.observe(lambda c: self._on_param_change(), names="value")
        self.tg_brpoint.observe(lambda c: self._draw_norm(), names="value")
        for t in self._all_toggles:
            t.observe(lambda c: self._draw(), names="value")

        self._render_loader(active=True)
        self._on_mode_change("apod")

    #
    def _msg(self, text, color="green"):
        with self.out_status:
            clear_output(wait=True)
            display(HTML(f"<span style='color:{color};font-weight:bold'>{text}</span>"))

    def _reset_current(self):
        # prevents window compounding
        self.spec.current_fft = FFTData(self.spec.fft.canal.copy(),
                                        self.spec.fft.values.copy())

    def _set_toggles_off(self):
        self._busy = True
        for t in self._all_toggles:
            t.value = False
        self._busy = False

    # file loading -- uses ipywidgets.FileUpload (works in plain Jupyter,
    # JupyterLab, Voila and Binder -- no google.colab dependency needed).
    def _make_uploader(self):
        uploader = widgets.FileUpload(
            accept=".txt,.csv,.dat,.asc,.jdx,.dx",
            multiple=False,
            description="Load file",
            button_style="primary",
        )
        uploader.observe(self._on_file_upload, names="value")
        return uploader

    def _render_loader(self, active):
        with self.out_loader:
            clear_output(wait=True)
            if active:
                self.uploader = self._make_uploader()
                display(self.uploader)
            else:
                display(HTML("""
                <span style="display:inline-block;background:#9e9e9e;color:#fff;padding:6px 18px;
                             border-radius:4px;font-weight:bold;font-family:sans-serif;cursor:not-allowed;">
                  File loaded (use Clear File)
                </span>"""))

    def _on_file_upload(self, change):
        if self.spec is not None:
            return
        new_value = change["new"]
        if not new_value:
            return

        # ipywidgets >=8 gives a tuple of dicts; ipywidgets 7 gives a dict
        # keyed by filename. Handle both so this works regardless of the
        # exact version Binder/JupyterHub resolves.
        if isinstance(new_value, dict):
            name, fileinfo = next(iter(new_value.items()))
            content = fileinfo["content"]
        else:
            fileinfo = new_value[0]
            name = fileinfo["name"]
            content = fileinfo["content"]

        try:
            raw = bytes(content)
            x, y, meta = load_spectrum(raw, name)
            self.spec = SpectrumData(np.asarray(x, float), np.asarray(y, float), ProcessConfig())
            self.spec.preprocess()
            self.spec.compute_fft()
            self.spec.compute_log_fft()
            self._applied = False
            self.lbl_file.value = f"Loaded: {name}  ({len(x)} points)"
            self._render_loader(active=False)
            self.btn_clear.disabled = False
            for t in self._proc_toggles:
                t.disabled = True
            self.btn_export.disabled = True
            self.tg_orig.value = True
            self._msg(f"Loaded: {name}", "green")
            self._draw()
            self._draw_norm()
        except Exception as e:
            self._msg(f"Load failed: {e}", "red")

    def _clear(self, _):
        # Reset everything and re-enable loading
        self.spec = None
        self._applied = False
        self.lbl_file.value = "No file loaded"
        self._set_toggles_off()
        for t in self._proc_toggles:
            t.disabled = True
        self.btn_export.disabled = True
        self.btn_clear.disabled = True
        self._render_loader(active=True)
        with self.out_plot:
            clear_output(wait=False)
        with self.out_norm:
            clear_output(wait=False)
        with self.out_download:
            clear_output(wait=False)
        self._msg("Cleared. You can load a new file.", "purple")

    # mode, window, parameter controls
    def _on_mode_change(self, mode):
        self.mode = mode
        show = (mode == "deconv")
        self.sl_delta.layout.display = "flex" if show else "none"
        self.sl_rho.layout.display = "flex" if show else "none"
        self._draw_norm()

    def _on_window_change(self):
        # Choosing a window auto-shows LOG FFT and refreshes the panel.
        if self.spec is None:
            return
        self.tg_lfft.value = True
        self._draw()
        self._draw_norm()

    def _on_param_change(self):
        #update overlaid curves and the Apod.Point label.
        self.lbl_apod.value = f"Apod.Point = {int(round(self.sl_sigma.value*100))}%"
        self._draw_norm()

    def _toggle_panel(self, _):
        self._panel_open = not self._panel_open
        self.panel.layout.display = "flex" if self._panel_open else "none"
        self.btn_close.description = "Close" if self._panel_open else "Open"
        if self._panel_open:
            self._draw_norm()

    # apply processing
    def _apply(self, _):
        if self.spec is None:
            self._msg("Load a file first.", "red")
            return
        try:
            win, sigma = self.dd_win.value, self.sl_sigma.value
            if self.mode == "apod":
                self._reset_current()
                self.spec.deconv_filter = None
                self.spec.apodize(window_type=win, sigma=sigma)
            else:
                self.spec.deconvolve_with_apodization(
                    delta=self.sl_delta.value, rho=self.sl_rho.value,
                    window_type=win, sigma=sigma)
            self.spec.compute_ifft()
            self._applied = True
            for t in self._proc_toggles:
                t.disabled = False
            self.btn_export.disabled = False
            label = "Apodization" if self.mode == "apod" else "Deconvolution + Apodization"
            self._msg(f"{label} applied ('{win}').", "green")
            self.tg_mfft.value = True
            self._draw()
            self._draw_norm()
        except Exception as e:
            self._msg(f"Apply failed: {e}", "red")

    def _export(self, _):
        if self.spec is None or self.spec.ifft is None:
            self._msg("Apply a window first.", "red")
            return
        try:
            out = np.column_stack((self.spec.x, self.spec.ifft))
            buf = io.StringIO()
            np.savetxt(buf, out, fmt="%.6f", header="Wavenumber Intensity")
            filename = "processed_signal.txt"
            self._trigger_download(filename, buf.getvalue())
            self._msg(f"Exported {filename}", "green")
        except Exception as e:
            self._msg(f"Export failed: {e}", "red")

    def _trigger_download(self, filename, text_data):
        # Browser-side download that works the same in classic Jupyter,
        # JupyterLab, and Voila -- no google.colab dependency needed.
        b64 = base64.b64encode(text_data.encode()).decode()
        with self.out_download:
            clear_output(wait=True)
            display(HTML(f"""
            <a id="spectrumgui-dl" download="{filename}"
               href="data:text/plain;base64,{b64}" style="display:none">download</a>
            <script>
              document.getElementById('spectrumgui-dl').click();
            </script>
            """))

    #Overlaid curves, driven by the slider
    def _draw_norm(self):
        if self.spec is None or not self._panel_open:
            return
        s = self.spec
        if s.fft is None:
            s.compute_fft()
        if s.log_fft is None:
            s.compute_log_fft()
        canal = s.fft.canal
        canal_abs = np.abs(canal)
        m = canal > 0
        win = s.build_apod_window(canal, self.dd_win.value, None, self.sl_sigma.value)

        with self.out_norm:
            clear_output(wait=True)
            plt.close("all")
            fig, ax = plt.subplots(figsize=(9, 3.8))
            ax.plot(canal[m], s.log_fft[m], lw=0.8, label="log FFT")
            if self.mode == "apod":
                wdb = 10 * np.log10(np.maximum(win[m], 1e-12))
                ax.plot(canal[m], wdb, lw=1.2, ls="--", label="apodization curve")
            else:
                EPS = 1e-6
                H = np.exp(-self.sl_delta.value * (canal_abs ** self.sl_rho.value))
                dec = 1.0 / (H + 1e-12)
                comb = dec * win
                wdb = 20 * np.log10(np.maximum(win[m], EPS))
                cdb = 20 * np.log10(np.maximum(comb[m], EPS))
                ax.plot(canal[m], wdb, lw=1.2, ls="--", label="apodization curve")
                ax.plot(canal[m], cdb, lw=1.4, ls="-.", label="apodization + deconvolution curve")
                sig = s.log_fft[m]
                pad = (np.max(sig) - np.min(sig)) * 0.6
                ax.set_ylim(np.min(sig) - pad, np.max(sig) + pad)
            if self.tg_brpoint.value:
                L = self.sl_sigma.value * np.max(canal_abs)   # apodization break point
                ax.axvline(L, color="firebrick", ls=":")
            ax.set_title("Normalized Domain")
            ax.set_xlabel("Channel"); ax.set_ylabel("db")
            ax.legend(loc="upper right"); ax.grid()
            plt.show()

    # All other views
    def _draw(self):
        if self._busy or self.spec is None:
            return
        active = any(t.value for t in self._all_toggles)
        with self.out_plot:
            if not active:
                clear_output(wait=False)   # force-clear leftover plot
                return
            clear_output(wait=True)
            plt.close("all")
            s = self.spec
            if self.tg_orig.value:
                PlotDiagrams.plot_spectrum(s.x, s.y, "Original Signal")
            if self.tg_fft.value:
                PlotDiagrams.plot_fft(s.fft, title="FFT Spectrum")
            if self.tg_lfft.value:
                PlotDiagrams.plot_log_fft(s)
            if self.tg_mfft.value and self._applied:
                PlotDiagrams.plot_processed_fft(s, title="Modified FFT Signal")
            if self.tg_cmpfft.value and self._applied:
                PlotDiagrams.plot_fft_before_after_processing(s, title="Original vs Modified FFT")
            if self.tg_difffft.value and self._applied:
                PlotDiagrams.plot_processing_difference(s, title="Difference: Modified - Original FFT")
            if self.tg_msig.value and s.ifft is not None:
                PlotDiagrams.plot_ifft(s)
            if self.tg_cmpsig.value and s.ifft is not None:
                PlotDiagrams.plot_original_vs_ifft(s, title="Original and Modified Signal")
            if self.tg_diffsig.value and s.ifft is not None:
                PlotDiagrams.plot_ifft_difference(s, title="Difference: Original - Modified Signal")

    # layout
    def show(self):
        hr = lambda: widgets.HTML("<hr style='margin:6px 0'>")
        head = lambda t: widgets.HTML(f"<h3 style='margin:4px 0'>{t}</h3>")

        toolbar = widgets.HBox([self.btn_apply, self.tg_brpoint,
                                widgets.Label("Window:"), self.dd_win, self.btn_close])
        center = widgets.HBox([self.out_norm, widgets.VBox([self.sl_delta, self.sl_rho])])
        bottom = widgets.HBox([self.sl_sigma, self.lbl_apod])
        self.panel = widgets.VBox(
            [widgets.HTML("<b style='font-family:sans-serif'>Normalized Domain</b>"),
             toolbar, center, bottom],
            layout=widgets.Layout(border="1px solid #888", padding="6px", width="960px"))

        display(widgets.VBox([
            widgets.HTML("<h2>Spectrum Processing</h2>"),
            widgets.HBox([self.out_loader, self.btn_clear]),
            self.lbl_file, hr(),
            widgets.VBox([self.tg_orig, self.tg_fft, self.tg_lfft]), hr(),
            head("Apodization (signal processing)"),
            self.rb_mode,
            self.panel, hr(),
            head("Modified FFT Views"),
            widgets.HBox([self.tg_mfft, self.tg_cmpfft, self.tg_difffft]), hr(),
            head("Processed Signal Views"),
            widgets.HBox([self.tg_msig, self.tg_cmpsig, self.tg_diffsig]), hr(),
            self.btn_export, hr(),
            self.out_status, self.out_plot, self.out_download,
        ]))


In [ ]:
gui = SpectrumGUI()
gui.show()